In [1]:
import os
import re
import json
import datetime
import multiprocessing
import difflib
import logging
from functools import partial
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAIError, RateLimitError, APIError, Timeout
from anthropic import AnthropicError
# Add other provider-specific exceptions as needed
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_deepseek import ChatDeepSeek
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mistralai.chat_models import ChatMistralAI
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_xai import ChatXAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.exceptions import OutputParserException
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from datasets import load_dataset
from pathlib import Path
from collections import defaultdict

/home/tweichuan/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- Configuration ---

# Define paths for prompts (adjust as needed)
PROMPT_DIR = Path("./prompts/repair")
# SYSTEM_PROMPT_FILE = PROMPT_DIR / "system_prompt_clean.txt"
# USER_PROMPT_TEMPLATE_FILE = PROMPT_DIR / "user_prompt_template.txt"
# FIX_PATHS_JSON = Path("./ground_truth/bug_paths.json") # Path to your JSON file
# FIX_PATHS_JSON = Path("./localization_candidates/union_4voters_r1full.json") # Path to your JSON file
FIX_PATHS_JSON = Path("./localization_candidates/gemini_deepseekr1full_concise.json") # Path to your JSON file
CODEBASE_DIR = Path("./codebases")
EXPERIMENTS_BASE_DIR = Path("./repair_experiments")

# Hardcoded Configuration
CONFIG = {
    "fix_paths_json_path": str(FIX_PATHS_JSON),
}


# --- Logging Setup ---
logging.basicConfig(level=logging.WARN, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [3]:
def generate_combined_diff_json(
    experiment_time_id: str,
    diff_dir_name: str = "diffs", # Added: specify diff subdir name
    output_filename: str = "combined_patches.json",
    check_against_dataset: bool = True # Added: option to skip dataset check if not needed
    ):
    """
    Generates a JSON file containing aggregated diff patches for a given experiment run
    from a specified diff directory, and optionally prints instance_ids with no resulting diffs
    by checking against the SWE-bench Lite dataset.

    Args:
        experiment_time_id: The timestamp ID of the experiment run.
        diff_dir_name: The name of the subdirectory containing the diff files (e.g., "diffs", "reparsed_diffs").
        output_filename: The name for the output JSON file.
        check_against_dataset: Whether to load the dataset to report all missing instances.
    """
    experiment_path = EXPERIMENTS_BASE_DIR / experiment_time_id
    # Use the provided diff_dir_name
    diffs_dir = experiment_path / diff_dir_name
    config_path = experiment_path / "config.json"
    # Save output relative to the experiment path
    output_path = experiment_path / output_filename

    all_expected_instance_ids = set()
    if check_against_dataset:
        try:
            logger.info(f"Loading SWE-bench Lite dataset to get all instance IDs (for checking against {diff_dir_name})...")
            dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test", trust_remote_code=True)
            for item in dataset:
                all_expected_instance_ids.add(item['instance_id'])
            logger.info(f"Loaded {len(all_expected_instance_ids)} expected instance IDs from dataset.")
        except Exception as e:
            logger.error(f"Failed to load SWE-bench Lite dataset: {e}")
            logger.error("Reporting of missing instances might be incomplete.")
            check_against_dataset = False # Disable check if loading failed

    if not diffs_dir.is_dir():
        logger.error(f"Specified diffs directory not found for experiment '{experiment_time_id}': {diffs_dir}")
        if check_against_dataset and all_expected_instance_ids:
            logger.info(f"--- Instances with NO results (missing {diff_dir_name} directory) ---")
            for inst_id in sorted(list(all_expected_instance_ids)):
                print(f"  - {inst_id} ({diff_dir_name} directory not found)")
            logger.info("---------------------------------------------------------")
        return # Cannot proceed further

    # --- Try to get model name from config ---
    model_name_or_path = "unknown_model"
    try:
        # ... (rest of the model name loading logic remains the same) ...
        if config_path.is_file():
            with open(config_path, 'r', encoding='utf-8') as f:
                config_data = json.load(f)
                model_name_or_path = config_data.get("model_name", model_name_or_path)
        else:
            logger.warning(f"Config file not found at {config_path}. Using default model name for {output_filename}.")
    except Exception as e:
        logger.warning(f"Could not read model name from config file {config_path}: {e}. Using default.")

    # --- Aggregate diffs and track processed instances ---
    aggregated_diffs = defaultdict(str)
    instances_processed_with_dir = set()
    instances_with_empty_diffs = set()

    instance_dirs = [d for d in diffs_dir.iterdir() if d.is_dir()]

    logger.info(f"Aggregating diffs from {len(instance_dirs)} instance directories found in {diffs_dir}...")
    # ... (rest of the aggregation logic remains the same) ...
    for instance_dir in instance_dirs:
        instance_id = instance_dir.name
        instances_processed_with_dir.add(instance_id)

        diff_files = sorted(list(instance_dir.glob("*.diff")))

        if not diff_files:
            # logger.warning(f"No .diff files found in directory: {instance_dir}")
            instances_with_empty_diffs.add(instance_id)
            continue

        combined_patch_for_instance = ""
        valid_diff_found = False
        for diff_file in diff_files:
            try:
                with open(diff_file, 'r', encoding='utf-8') as f:
                    diff_content = f.read()
                    if diff_content.strip():
                        combined_patch_for_instance += diff_content
                        valid_diff_found = True
            except Exception as e:
                logger.error(f"Error reading diff file {diff_file} for instance {instance_id}: {e}")

        if valid_diff_found:
            aggregated_diffs[instance_id] = combined_patch_for_instance
        else:
            logger.warning(f"No non-empty diff content found for instance_id: {instance_id} in {instance_dir}")
            instances_with_empty_diffs.add(instance_id)


    # --- Identify ALL instances without successful diffs (Optional based on check_against_dataset) ---
    all_failed_or_missing_instances = set()
    if check_against_dataset and all_expected_instance_ids:
        instances_with_valid_diffs = set(aggregated_diffs.keys())
        all_failed_or_missing_instances = all_expected_instance_ids - instances_with_valid_diffs
    else:
        # Fallback or if check disabled: report only those processed but yielded empty diffs
        all_failed_or_missing_instances = instances_with_empty_diffs

    # --- Print Report (Optional based on check_against_dataset) ---
    if all_failed_or_missing_instances:
        logger.info(f"--- Instances with NO valid diff in '{diff_dir_name}' ({len(all_failed_or_missing_instances)} total) ---")
        # ... (rest of the printing logic remains the same) ...
        for inst_id in sorted(list(all_failed_or_missing_instances)):
            reason = ""
            if check_against_dataset and all_expected_instance_ids:
                if inst_id not in instances_processed_with_dir:
                    reason = f"({diff_dir_name}/{inst_id} dir not found)"
                elif inst_id in instances_with_empty_diffs:
                    reason = "(No diff files or only empty diffs found)"
            elif inst_id in instances_with_empty_diffs:
                reason = "(No diff files or only empty diffs found)"
            print(f"  - {inst_id} {reason}")
        logger.info("-------------------------------------------------------------------")

    else:
        logger.info(f"All processed instance directories in '{diff_dir_name}' seem to have yielded valid diffs.")


    # --- Format output JSON ---
    output_data = []
    # ... (rest of the formatting logic remains the same) ...
    for instance_id in sorted(aggregated_diffs.keys()):
        output_data.append({
            "instance_id": instance_id,
            "model_patch": aggregated_diffs[instance_id],
            "model_name_or_path": model_name_or_path
        })

    # --- Save the combined JSON file ---
    try:
        # ... (rest of the saving logic remains the same, using output_path) ...
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(output_data, f, indent=2)
        logger.info(f"Successfully generated combined patch file: {output_path} (contains {len(output_data)} instances from '{diff_dir_name}')")

    except Exception as e:
        logger.error(f"Error writing combined JSON file to {output_path}: {e}")

In [4]:
def load_json(file_path: Path) -> dict:
    """Loads a JSON file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        logger.error(f"JSON file not found: {file_path}")
        return {} # Return empty dict if not found, or raise error?
    except json.JSONDecodeError:
        logger.error(f"Error decoding JSON from {file_path}")
        return {}
    except Exception as e:
        logger.error(f"Error loading JSON from {file_path}: {e}")
        raise

def to_valid_file_name(path_str: str) -> str:
    """Converts a path string to a valid filename."""
    # Replace slashes and other problematic characters
    s = re.sub(r'[\\/*?:"<>|]', '_', path_str)
    # Optional: Truncate if too long
    max_len = 100
    if len(s) > max_len:
        s = s[:max_len]
    return s

def get_file_content(instance_id: str, relative_path: str) -> str | None:
    """Reads the content of a specific file within the instance's codebase."""
    full_path = CODEBASE_DIR / instance_id / relative_path
    try:
        with open(full_path, 'r', encoding='utf-8') as f:
            return f.read()
    except FileNotFoundError:
        logger.warning(f"File not found for {instance_id}: {full_path}")
        return None
    except Exception as e:
        logger.error(f"Error reading file {full_path}: {e}")
        return None

In [5]:
def parse_search_replace_xml(response_content: str, instance_id: str = None) -> list[tuple[str, str]]:
    """
    解析LLM回應以提取<original_code_N>和<fixed_code_N>區塊對。
    
    策略1: 使用正則表達式尋找完整的標籤對
    策略2: 處理只有起始標籤的情況
    
    返回: [(search_block, replace_block), ...]形式的元組列表。
    """
    pairs = []
    
    # 策略1: 尋找完整標籤對
    pattern_complete = re.compile(
        r"<original_code_(\d+)>([\s\S]*?)</original_code_\1>[\s\S]*?"
        r"<fixed_code_\1>([\s\S]*?)</fixed_code_\1>", 
        re.MULTILINE
    )
    
    matches = pattern_complete.findall(response_content)
    if matches:
        for match in matches:
            tag_num, original_code, fixed_code = match[0], match[1], match[2]
            
            # 清理代碼塊
            original_code = original_code.strip()
            fixed_code = fixed_code.strip()
            
            # 移除可能的```標記
            if original_code.startswith("```"):
                original_code = original_code[3:]
            if original_code.endswith("```"):
                original_code = original_code[:-3]
                
            if fixed_code.startswith("```"):
                fixed_code = fixed_code[3:]
            if fixed_code.endswith("```"):
                fixed_code = fixed_code[:-3]
                
            pairs.append((original_code.strip(), fixed_code.strip()))
        
        return pairs
    
    # 策略2: 處理只有起始標籤的情況
    pattern_open_tags = re.compile(r"<(original_code|fixed_code)_(\d+)>", re.MULTILINE)
    open_tags = [(match.group(1), int(match.group(2)), match.start(), match.end()) 
                for match in pattern_open_tags.finditer(response_content)]
    
    if open_tags:
        # 按位置排序標籤
        open_tags.sort(key=lambda x: x[2])
        
        # 處理標籤對
        i = 0
        while i < len(open_tags) - 1:
            current_tag = open_tags[i]
            next_tag = open_tags[i+1]
            
            # 檢查是否為原始代碼和修復代碼對
            if (current_tag[0] == "original_code" and 
                next_tag[0] == "fixed_code" and 
                current_tag[1] == next_tag[1]):  # 確認標籤編號相同
                
                # 提取代碼塊
                start_pos = current_tag[3]  # 原始代碼開始位置
                end_pos = next_tag[2]  # 修復代碼標籤開始位置
                original_code = response_content[start_pos:end_pos].strip()
                
                # 處理修復代碼塊
                next_end_pos = len(response_content)
                if i + 2 < len(open_tags):
                    next_end_pos = open_tags[i+2][2]
                fixed_code = response_content[next_tag[3]:next_end_pos].strip()
                
                # 清理代碼塊，移除```
                if original_code.startswith("```"):
                    original_code = original_code[3:]
                if original_code.endswith("```"):
                    original_code = original_code[:-3]
                    
                if fixed_code.startswith("```"):
                    fixed_code = fixed_code[3:]
                if fixed_code.endswith("```"):
                    fixed_code = fixed_code[:-3]
                
                pairs.append((original_code.strip(), fixed_code.strip()))
                i += 2  # 跳到下一對標籤
            else:
                i += 1  # 移動到下一個標籤
        
        if pairs:
            return pairs
    
    # 所有策略都失敗了
    if instance_id:
        print(f"無法解析回應，Instance ID: {instance_id}")
    return []

In [6]:
def reparse_and_regenerate_diffs(experiment_time_id: str, regenerate_combined: bool = True):
    """
    Reparses all raw responses for a given experiment, regenerates individual diffs
    into a new directory, and optionally regenerates the combined patch JSON file.

    Requires access to original codebases, fix_paths.json, and helper functions like
    get_file_content, parse_search_replace_xml, to_valid_file_name, load_json.
    """
    logger.info(f"Starting reparse and regeneration for experiment: {experiment_time_id}")
    experiment_path = EXPERIMENTS_BASE_DIR / experiment_time_id
    raw_responses_dir = experiment_path / "raw_responses"
    # Define a *new* directory for the regenerated diffs
    reparsed_diffs_dir = experiment_path / "reparsed_diffs"
    fix_paths_file = Path(CONFIG["fix_paths_json_path"]) # Get path from config

    # --- Pre-checks ---
    if not raw_responses_dir.is_dir():
        logger.error(f"Raw responses directory not found: {raw_responses_dir}")
        return
    if not CODEBASE_DIR.is_dir(): # Assuming CODEBASE_DIR is defined globally
        logger.error(f"Codebases directory not found: {CODEBASE_DIR}")
        return
    if not fix_paths_file.is_file():
        logger.error(f"Fix paths JSON file not found: {fix_paths_file}")
        return

    # --- Load necessary data ---
    try:
        fix_paths_data = load_json(fix_paths_file) # Assumes load_json is available
        if not fix_paths_data:
             logger.error(f"Fix paths data is empty in {fix_paths_file}. Cannot proceed.")
             return
    except Exception as e:
        logger.error(f"Failed to load fix paths data: {e}")
        return

    # Load dataset to iterate through instances (more robust)
    all_instance_ids = set()
    try:
        dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test", trust_remote_code=True)
        all_instance_ids = {item['instance_id'] for item in dataset}
        logger.info(f"Loaded {len(all_instance_ids)} instance IDs from dataset for reparsing.")
    except Exception as e:
        logger.error(f"Failed to load dataset for instance list: {e}. Reparsing may be incomplete.")
        # Decide whether to proceed based on found raw response dirs? For now, let's stop.
        return


    # --- Ensure output directory exists and is empty? ---
    try:
        reparsed_diffs_dir.mkdir(parents=True, exist_ok=True)
        # Optional: Clean the directory before starting?
        # import shutil
        # shutil.rmtree(reparsed_diffs_dir)
        # reparsed_diffs_dir.mkdir(parents=True, exist_ok=True)
        logger.info(f"Output directory for reparsed diffs: {reparsed_diffs_dir}")
    except Exception as e:
        logger.error(f"Failed to create reparsed diffs directory: {e}")
        return

    # --- Iterate through instances and paths ---
    total_files_processed = 0
    diffs_regenerated_count = 0
    blocks_not_found_count = 0
    blocks_fixed_by_trimming = 0

    # 新增：一個輔助函數，用於檢查並刪除頭尾相同的行
    def try_trim_blocks(search_block, replace_block, original_content):
        """嘗試通過刪除頭尾相同的行來找到代碼塊。返回 (是否找到, 修改後的內容)"""
        # 將代碼塊拆分為行
        search_lines = search_block.splitlines()
        replace_lines = replace_block.splitlines()
        
        # 如果任一塊為空或只有一行，無法刪除
        if len(search_lines) <= 1 or len(replace_lines) <= 1:
            return False, original_content
        
        current_search = search_block
        current_replace = replace_block
        lines_trimmed = 0
        
        # 不斷嘗試刪除頭尾
        while len(search_lines) > 1 and len(replace_lines) > 1:
            found = False
            modified = original_content
            
            # 檢查第一行是否相同
            if search_lines[0] == replace_lines[0]:
                # 刪除第一行
                search_lines = search_lines[1:]
                replace_lines = replace_lines[1:]
                lines_trimmed += 1
                current_search = "\n".join(search_lines)
                current_replace = "\n".join(replace_lines)
                
                # 檢查是否找到
                if current_search in original_content:
                    try:
                        modified = original_content.replace(current_search, current_replace, 1)
                        return True, modified
                    except Exception:
                        pass
                    
            # 如果第一行不同或刪除後仍未找到，則檢查最後一行
            elif search_lines[-1] == replace_lines[-1]:
                # 刪除最後一行
                search_lines = search_lines[:-1]
                replace_lines = replace_lines[:-1]
                lines_trimmed += 1
                current_search = "\n".join(search_lines)
                current_replace = "\n".join(replace_lines)
                
                # 檢查是否找到
                if current_search in original_content:
                    try:
                        modified = original_content.replace(current_search, current_replace, 1)
                        return True, modified
                    except Exception:
                        pass
            else:
                # 如果頭尾都不同，無法繼續刪除
                break
        
        # 如果刪到不能再刪，仍未找到
        return False, original_content

    # Use tqdm for progress if dataset is large
    for instance_id in tqdm(sorted(list(all_instance_ids)), desc="Reparsing Instances"):
        # if instance_id != "django__django-16910":
        #     continue
        # else:
        #     print("hahahahahahahahahahaha")


        paths_to_fix = fix_paths_data.get(instance_id)
        if type(paths_to_fix) == str:
            paths_to_fix = [paths_to_fix]

        if not paths_to_fix:
            logger.info(f"No fix paths defined for instance {instance_id}, skipping.")
            continue

        instance_raw_dir = raw_responses_dir / instance_id
        instance_reparsed_diff_dir = reparsed_diffs_dir / instance_id

        if not instance_raw_dir.is_dir():
            logger.info(f"No raw responses found for instance {instance_id}, skipping.")
            continue

        instance_reparsed_diff_dir.mkdir(parents=True, exist_ok=True)

        for path in paths_to_fix:
            total_files_processed += 1
            raw_response_filename = f"{to_valid_file_name(path)}.txt" # Assumes to_valid_file_name available
            raw_response_path = instance_raw_dir / raw_response_filename

            if not raw_response_path.is_file():
                logger.info(f"Raw response file not found for {instance_id}/{path}, skipping.")
                continue

            # --- Read Raw Response ---
            try:
                with open(raw_response_path, 'r', encoding='utf-8') as f:
                    response_content = f.read()
            except Exception as e:
                logger.error(f"Error reading raw response {raw_response_path}: {e}")
                continue

            # --- Reparse ---
            # Assumes parse_search_replace_xml is available
            repair_pairs = parse_search_replace_xml(response_content, instance_id)
            
            # print("REPAIR_PAIRS ------------------------")
            # print(repair_pairs)
            # print("------------------------ REPAIR_PAIRS") 

            if not repair_pairs:
                logger.info(f"Reparsing yielded no pairs for {instance_id}/{path}.")
                continue

            # --- Get Original Content ---
            # Assumes get_file_content is available
            codebase_file_path = CODEBASE_DIR / instance_id / path
            original_content = get_file_content(instance_id, path)
            if original_content is None:
                logger.error(f"Could not get original content for {instance_id}/{path} during reparse.")
                continue

            # --- Apply Changes Sequentially ---
            modified_content = original_content
            applied_changes_count = 0
            
            # (Use the same sequential application logic as in repair_file)
            for i, (search_block, replace_block) in enumerate(repair_pairs):
                if search_block == replace_block: 
                    continue
                
                if search_block in modified_content:
                    try:
                        modified_content = modified_content.replace(search_block, replace_block, 1)
                        applied_changes_count += 1
                    except Exception as e:
                        logger.error(f"Error applying reparsed change {i+1} for {instance_id}/{path}: {e}")
                        applied_changes_count = 0 # Mark as failed
                        break
                else:
                    # 嘗試通過刪除頭尾相同行來修正代碼塊
                    found, new_content = try_trim_blocks(search_block, replace_block, modified_content)
                    
                    if found:
                        # 成功找到並應用了修改
                        blocks_fixed_by_trimming += 1
                        modified_content = new_content
                        applied_changes_count += 1
                        logger.info(f"Successfully fixed block by trimming for {instance_id}/{path}")
                    else:
                        # 仍然找不到，印出找不到代碼塊的詳細資訊
                        blocks_not_found_count += 1
                        # print(f"\n====== BLOCK NOT FOUND ======")
                        # print(f"Instance ID: {instance_id}")
                        # print(f"Codebase path: ./{codebase_file_path}")
                        # print(f"Raw response path: ./{raw_response_path}")
                        # print(f"\n--- Search Block (not found) ---\n{search_block}")
                        # print(f"\n--- Replace Block ---\n{replace_block}")
                        # print(f"===============================\n")
                        
                        logger.warning(f"Reparsed search block {i+1} not found in current content for {instance_id}/{path}, even after trimming.")
                        # 繼續處理下一個代碼塊，不中斷執行

            if applied_changes_count == 0:
                # logger.info(f"Reparsing applied no effective changes for {instance_id}/{path}.")
                # print("NO ANY CHANGES")
                continue # Skip generating diff if no changes

            # --- Regenerate Diff ---
            reparsed_diff_path = instance_reparsed_diff_dir / f"{to_valid_file_name(path)}.diff"
            try:
                # Assumes difflib is imported
                diff = difflib.unified_diff(
                    original_content.splitlines(keepends=True),
                    modified_content.splitlines(keepends=True),
                    fromfile=f"a/{path}",
                    tofile=f"b/{path}",
                )
                diff_str = "".join(diff)

                if diff_str.strip(): # Save only if diff is non-empty
                    with open(reparsed_diff_path, "w", encoding="utf-8") as f:
                        f.write(diff_str)
                    diffs_regenerated_count += 1
                # else:
                    # logger.info(f"Regenerated diff was empty for {instance_id}/{path}.")

            except Exception as e:
                logger.error(f"Failed to regenerate or save diff for {instance_id}/{path}: {e}")

    logger.info(f"Reparsing complete. Processed {total_files_processed} potential files. Regenerated {diffs_regenerated_count} non-empty diff files in {reparsed_diffs_dir}.")
    print(f"Total blocks not found in original content: {blocks_not_found_count}")
    print(f"Total blocks fixed by trimming: {blocks_fixed_by_trimming}")

    # --- Optionally Regenerate Combined JSON ---
    if regenerate_combined:
        logger.info("Regenerating combined patch JSON from reparsed diffs...")
        try:
            # Call the modified generator function, pointing to the new diff directory
            generate_combined_diff_json(
                experiment_time_id,
                diff_dir_name="reparsed_diffs", # Read from the new directory
                output_filename="reparsed_combined_patches.json"
            )
        except Exception as e:
            logger.error(f"Failed to regenerate combined JSON: {e}")

In [7]:
# def reparse_and_regenerate_diffs(experiment_time_id: str, regenerate_combined: bool = True):
#     """
#     Reparses all raw responses for a given experiment, regenerates individual diffs
#     into a new directory, and optionally regenerates the combined patch JSON file.

#     Requires access to original codebases, fix_paths.json, and helper functions like
#     get_file_content, parse_search_replace_xml, to_valid_file_name, load_json.
#     """
#     logger.info(f"Starting reparse and regeneration for experiment: {experiment_time_id}")
#     experiment_path = EXPERIMENTS_BASE_DIR / experiment_time_id
#     raw_responses_dir = experiment_path / "raw_responses"
#     reparsed_diffs_dir = experiment_path / "reparsed_diffs" # Define a *new* directory
#     fix_paths_file = Path(CONFIG["fix_paths_json_path"])

#     if not raw_responses_dir.is_dir():
#         logger.error(f"Raw responses directory not found: {raw_responses_dir}")
#         return
#     if not CODEBASE_DIR.is_dir():
#         logger.error(f"Codebases directory not found: {CODEBASE_DIR}")
#         return
#     if not fix_paths_file.is_file():
#         logger.error(f"Fix paths JSON file not found: {fix_paths_file}")
#         return

#     try:
#         fix_paths_data = load_json(fix_paths_file)
#         if not fix_paths_data:
#              logger.error(f"Fix paths data is empty in {fix_paths_file}. Cannot proceed.")
#              return
#     except Exception as e:
#         logger.error(f"Failed to load fix paths data: {e}")
#         return

#     all_instance_ids = set()
#     try:
#         dataset = load_dataset("princeton-nlp/SWE-bench_Lite", split="test", trust_remote_code=True)
#         all_instance_ids = {item['instance_id'] for item in dataset}
#         logger.info(f"Loaded {len(all_instance_ids)} instance IDs from dataset for reparsing.")
#     except Exception as e:
#         logger.error(f"Failed to load dataset for instance list: {e}. Reparsing may be incomplete.")
#         return

#     try:
#         reparsed_diffs_dir.mkdir(parents=True, exist_ok=True)
#         logger.info(f"Output directory for reparsed diffs: {reparsed_diffs_dir}")
#     except Exception as e:
#         logger.error(f"Failed to create reparsed diffs directory: {e}")
#         return

#     total_files_processed = 0
#     diffs_regenerated_count = 0
#     blocks_not_found_count = 0
#     blocks_fixed_by_trimming = 0
#     # New counter for ambiguous blocks after trimming
#     ambiguous_blocks_after_trimming_count = 0


#     # Modified try_trim_blocks function
#     def try_trim_blocks(search_block, replace_block, original_content, instance_id_for_log=None, path_for_log=None):
#         """
#         Tries to find the code block by trimming identical leading/trailing lines.
#         Returns (found_uniquely, modified_content, was_ambiguous_at_some_point).
#         `was_ambiguous_at_some_point` is True if any trimmed version was found more than once.
#         """
#         search_lines = search_block.splitlines()
#         replace_lines = replace_block.splitlines()
        
#         # If either block is empty or has only one line, trimming isn't meaningful here or possible.
#         if not search_lines or not replace_lines or len(search_lines) <= 1 or len(replace_lines) <= 1 :
#             return False, original_content, False

#         _search_lines = list(search_lines) # work with copies
#         _replace_lines = list(replace_lines)
        
#         initial_search_len = len(_search_lines)
#         initial_replace_len = len(_replace_lines)
        
#         lines_trimmed_total = 0
#         ever_ambiguous = False # Track if any trimmed version was ambiguous

#         # Iteratively trim as long as blocks are non-trivial and trimmable
#         while len(_search_lines) > 1 and len(_replace_lines) > 1:
#             trimmed_in_this_iteration = False
            
#             # Try trimming the first line
#             if _search_lines[0] == _replace_lines[0]:
#                 _search_lines.pop(0)
#                 _replace_lines.pop(0)
#                 lines_trimmed_total +=1
#                 trimmed_in_this_iteration = True
                
#                 current_search_str = "\n".join(_search_lines)
#                 if not current_search_str: # Avoid empty search string
#                     _search_lines.insert(0, search_lines[lines_trimmed_total-1]) # backtrack last trim
#                     _replace_lines.insert(0, replace_lines[lines_trimmed_total-1])
#                     lines_trimmed_total -=1
#                     trimmed_in_this_iteration = False # effectively cancelled this trim
#                     break # Stop if search block becomes empty

#                 if current_search_str in original_content:
#                     count = original_content.count(current_search_str)
#                     if count == 1:
#                         current_replace_str = "\n".join(_replace_lines)
#                         try:
#                             modified = original_content.replace(current_search_str, current_replace_str, 1)
#                             return True, modified, ever_ambiguous
#                         except Exception as e:
#                             logger.debug(f"Error during replacement after head trim for {instance_id_for_log}/{path_for_log}: {e}")
#                             # Treat as not found and let logic continue or break
#                     elif count > 1:
#                         ever_ambiguous = True
#                         logger.debug(f"Ambiguous match after head trim for {instance_id_for_log}/{path_for_log}. "
#                                      f"Block '{current_search_str[:50]}...' found {count} times. Continuing to trim.")
#                         # Don't return, continue trimming with the smaller block
#                     # If count == 0 (after 'in' check, implies block was just context lines) or ambiguous, continue trimming
            
#             # Try trimming the last line (only if still valid and head trim didn't result in a return)
#             # And only if something was trimmed at the head or if this is the first attempt in the loop
#             # to avoid an infinite loop if neither head nor tail can be trimmed further.
#             if len(_search_lines) > 1 and len(_replace_lines) > 1 and _search_lines[-1] == _replace_lines[-1]:
#                 _search_lines.pop(-1)
#                 _replace_lines.pop(-1)
#                 lines_trimmed_total +=1
#                 trimmed_in_this_iteration = True

#                 current_search_str = "\n".join(_search_lines)
#                 if not current_search_str: # Avoid empty search string
#                     _search_lines.append(search_lines[initial_search_len - (lines_trimmed_total - len(_search_lines))]) # backtrack
#                     _replace_lines.append(replace_lines[initial_replace_len - (lines_trimmed_total - len(_replace_lines))])
#                     lines_trimmed_total -=1
#                     trimmed_in_this_iteration = False
#                     break # Stop if search block becomes empty

#                 if current_search_str in original_content:
#                     count = original_content.count(current_search_str)
#                     if count == 1:
#                         current_replace_str = "\n".join(_replace_lines)
#                         try:
#                             modified = original_content.replace(current_search_str, current_replace_str, 1)
#                             return True, modified, ever_ambiguous
#                         except Exception as e:
#                             logger.debug(f"Error during replacement after tail trim for {instance_id_for_log}/{path_for_log}: {e}")
#                     elif count > 1:
#                         ever_ambiguous = True
#                         logger.debug(f"Ambiguous match after tail trim for {instance_id_for_log}/{path_for_log}. "
#                                      f"Block '{current_search_str[:50]}...' found {count} times. Continuing to trim.")
            
#             if not trimmed_in_this_iteration:
#                 # If no lines could be trimmed from head or tail in this iteration, break.
#                 break
        
#         # If loop finishes without a unique match
#         return False, original_content, ever_ambiguous


#     for instance_id in tqdm(sorted(list(all_instance_ids)), desc="Reparsing Instances"):
#         paths_to_fix = fix_paths_data.get(instance_id)
#         if type(paths_to_fix) == str:
#             paths_to_fix = [paths_to_fix]

#         if not paths_to_fix:
#             logger.debug(f"No fix paths defined for instance {instance_id}, skipping.")
#             continue

#         instance_raw_dir = raw_responses_dir / instance_id
#         instance_reparsed_diff_dir = reparsed_diffs_dir / instance_id

#         if not instance_raw_dir.is_dir():
#             logger.debug(f"No raw responses found for instance {instance_id}, skipping.")
#             continue

#         instance_reparsed_diff_dir.mkdir(parents=True, exist_ok=True)

#         for path in paths_to_fix:
#             total_files_processed += 1
#             raw_response_filename = f"{to_valid_file_name(path)}.txt"
#             raw_response_path = instance_raw_dir / raw_response_filename

#             if not raw_response_path.is_file():
#                 logger.debug(f"Raw response file not found for {instance_id}/{path}, skipping.")
#                 continue

#             try:
#                 with open(raw_response_path, 'r', encoding='utf-8') as f:
#                     response_content = f.read()
#             except Exception as e:
#                 logger.error(f"Error reading raw response {raw_response_path}: {e}")
#                 continue

#             repair_pairs = parse_search_replace_xml(response_content, instance_id)
#             if not repair_pairs:
#                 logger.debug(f"Reparsing yielded no pairs for {instance_id}/{path}.")
#                 continue

#             original_content = get_file_content(instance_id, path)
#             if original_content is None:
#                 logger.error(f"Could not get original content for {instance_id}/{path} during reparse.")
#                 continue

#             modified_content = original_content
#             applied_changes_count = 0
            
#             for i, (search_block, replace_block) in enumerate(repair_pairs):
#                 if search_block == replace_block: 
#                     continue
                
#                 if search_block in modified_content:
#                     # Check for ambiguity of the *original* untrimmed search_block
#                     if modified_content.count(search_block) > 1:
#                         logger.warning(f"Original search block {i+1} is ambiguous (found {modified_content.count(search_block)} times) "
#                                        f"for {instance_id}/{path}. Skipping this specific replacement.")
#                         # Potentially try trimming even for original ambiguous blocks?
#                         # For now, if original is ambiguous, we skip it as per "视为失败" if >1.
#                         # If we want to try trimming, we'd move this logic into an else for the `try_trim_blocks` call.
#                         # However, the request was about ambiguity *after* trimming.
#                         # So, if the original untrimmed block is ambiguous, it's a different kind of failure.
#                         # Let's assume for now, if original block is ambiguous, we don't attempt to fix.
#                         # This could be a policy decision.
#                         # The problem statement was "if in original code ... *a replace code block* ... >1 ... 失败"
#                         # This implies the *search_block that is about to be replaced*.
#                         blocks_not_found_count += 1 # Counting this as a type of "not found" or "failed to apply"
#                         continue # Skip this ambiguous block

#                     try:
#                         modified_content = modified_content.replace(search_block, replace_block, 1)
#                         applied_changes_count += 1
#                     except Exception as e:
#                         logger.error(f"Error applying reparsed change {i+1} for {instance_id}/{path}: {e}")
#                         # Consider if we should break or continue with next pair
#                         break 
#                 else: # Search block not found directly, try trimming
#                     found_by_trimming, temp_modified_content, was_ambiguous = try_trim_blocks(
#                         search_block, replace_block, modified_content,
#                         instance_id_for_log=instance_id, path_for_log=path
#                     )
                    
#                     if found_by_trimming:
#                         blocks_fixed_by_trimming += 1
#                         modified_content = temp_modified_content
#                         applied_changes_count += 1
#                         logger.info(f"Successfully applied trimmed block for {instance_id}/{path}")
#                         if was_ambiguous:
#                              # This means at some point during trimming, an intermediate form was ambiguous
#                              # but further trimming resolved it to a unique match.
#                              logger.debug(f"Block for {instance_id}/{path} was ambiguous at some trim level but resolved.")
#                     else: # Not found even after trimming attempts
#                         blocks_not_found_count += 1
#                         if was_ambiguous: # This means the best it could do was an ambiguous trimmed block
#                             ambiguous_blocks_after_trimming_count +=1
#                             logger.warning(f"Reparsed search block {i+1} not found for {instance_id}/{path}. "
#                                          f"Trimming led to ambiguous matches or no match.")
#                         else: # Not found and was never ambiguous during trimming
#                              logger.warning(f"Reparsed search block {i+1} not found for {instance_id}/{path}, "
#                                          f"even after trimming (no ambiguity encountered during trim).")
#                         # Log details for manual inspection if needed
#                         # logger.debug(f"Details for block not found: Instance: {instance_id}, Path: {path}, Raw Response: {raw_response_path}")
#                         # logger.debug(f"--- Search Block (not found) ---\n{search_block}\n--- End Search Block ---")
#                         # Continue processing other pairs for this file

#             if applied_changes_count == 0 and original_content == modified_content:
#                 logger.debug(f"Reparsing applied no effective changes for {instance_id}/{path}.")
#                 continue

#             reparsed_diff_path = instance_reparsed_diff_dir / f"{to_valid_file_name(path)}.diff"
#             try:
#                 diff = difflib.unified_diff(
#                     original_content.splitlines(keepends=True),
#                     modified_content.splitlines(keepends=True),
#                     fromfile=f"a/{path}",
#                     tofile=f"b/{path}",
#                 )
#                 diff_str = "".join(diff)

#                 if diff_str.strip():
#                     with open(reparsed_diff_path, "w", encoding="utf-8") as f:
#                         f.write(diff_str)
#                     diffs_regenerated_count += 1
#                 else:
#                     logger.debug(f"Regenerated diff was empty for {instance_id}/{path} despite content change (likely whitespace).")
#             except Exception as e:
#                 logger.error(f"Failed to regenerate or save diff for {instance_id}/{path}: {e}")

#     logger.info(f"Reparsing complete. Processed {total_files_processed} potential files.")
#     logger.info(f"Regenerated {diffs_regenerated_count} non-empty diff files in {reparsed_diffs_dir}.")
#     logger.info(f"Total blocks not found (or ambiguous original): {blocks_not_found_count}")
#     logger.info(f"Total blocks fixed by trimming (unambiguously): {blocks_fixed_by_trimming}")
#     logger.info(f"Total blocks where trimming resulted in ambiguity or no match after being ambiguous: {ambiguous_blocks_after_trimming_count}")


#     if regenerate_combined:
#         logger.info("Regenerating combined patch JSON from reparsed diffs...")
#         try:
#             generate_combined_diff_json(
#                 experiment_time_id,
#                 diff_dir_name="reparsed_diffs",
#                 output_filename="reparsed_combined_patches.json"
#             )
#         except Exception as e:
#             logger.error(f"Failed to regenerate combined JSON: {e}")

In [8]:
time_id = "20250510_021025"
reparse_and_regenerate_diffs(time_id, regenerate_combined=True)

Reparsing Instances:  19%|█▉        | 58/300 [00:00<00:01, 128.25it/s]2025-05-10 02:38:13,479 - WARNING - Reparsed search block 2 not found in current content for django__django-13710/tests/admin_inlines/tests.py, even after trimming.
2025-05-10 02:38:13,505 - WARNING - Reparsed search block 6 not found in current content for django__django-13964/django/db/models/fields/related.py, even after trimming.
Reparsing Instances:  25%|██▌       | 76/300 [00:00<00:01, 142.54it/s]2025-05-10 02:38:13,585 - WARNING - Reparsed search block 1 not found in current content for django__django-14787/tests/decorators/tests.py, even after trimming.
2025-05-10 02:38:13,643 - WARNING - Reparsed search block 1 not found in current content for django__django-15252/django/db/backends/base/creation.py, even after trimming.
Reparsing Instances:  31%|███▏      | 94/300 [00:00<00:01, 153.84it/s]2025-05-10 02:38:13,698 - WARNING - Reparsed search block 1 not found in current content for django__django-15814/django

無法解析回應，Instance ID: django__django-14672


2025-05-10 02:38:13,825 - WARNING - Reparsed search block 1 not found in current content for django__django-16910/django/db/models/fields/related.py, even after trimming.
2025-05-10 02:38:13,828 - WARNING - Reparsed search block 2 not found in current content for django__django-16910/django/db/models/fields/related.py, even after trimming.
2025-05-10 02:38:13,851 - WARNING - Reparsed search block 3 not found in current content for django__django-17087/django/db/migrations/writer.py, even after trimming.
Reparsing Instances:  43%|████▎     | 129/300 [00:00<00:01, 121.45it/s]2025-05-10 02:38:14,026 - WARNING - Reparsed search block 1 not found in current content for matplotlib__matplotlib-25079/lib/matplotlib/colors.py, even after trimming.


無法解析回應，Instance ID: django__django-17051


Reparsing Instances:  48%|████▊     | 143/300 [00:01<00:01, 98.71it/s] 2025-05-10 02:38:14,206 - WARNING - Reparsed search block 1 not found in current content for mwaskom__seaborn-2848/seaborn/relational.py, even after trimming.
2025-05-10 02:38:14,270 - WARNING - Reparsed search block 1 not found in current content for psf__requests-2148/requests/packages/urllib3/response.py, even after trimming.
2025-05-10 02:38:14,284 - WARNING - Reparsed search block 5 not found in current content for psf__requests-2674/requests/sessions.py, even after trimming.
2025-05-10 02:38:14,303 - WARNING - Reparsed search block 1 not found in current content for pydata__xarray-3364/xarray/core/dataset.py, even after trimming.
Reparsing Instances:  52%|█████▏    | 157/300 [00:01<00:01, 99.20it/s]2025-05-10 02:38:14,388 - WARNING - Reparsed search block 2 not found in current content for pydata__xarray-4493/xarray/core/merge.py, even after trimming.
2025-05-10 02:38:14,440 - WARNING - Reparsed search block 1

無法解析回應，Instance ID: sympy__sympy-20442
Total blocks not found in original content: 41
Total blocks fixed by trimming: 37


  - django__django-16400 (reparsed_diffs/django__django-16400 dir not found)
  - pylint-dev__pylint-7228 (reparsed_diffs/pylint-dev__pylint-7228 dir not found)
  - pytest-dev__pytest-11148 (reparsed_diffs/pytest-dev__pytest-11148 dir not found)
  - sympy__sympy-13895 (reparsed_diffs/sympy__sympy-13895 dir not found)
  - sympy__sympy-18621 (reparsed_diffs/sympy__sympy-18621 dir not found)
